# Phase 7 PCR: LoRA, LoRMA, full fine-tuning and distillation on my phase 1 gqa_swa model

Attach the phase 1 output (`gqa_swa_best.pt` and `bpe_legal_16k_v2.pkl`) and add the `HF` secret.

In [ ]:
import os, gc, re, json, math, time, pickle, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for secret_name in ("HF_TOKEN", "HF"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(secret_name)
            if HF_TOKEN:
                break
        except Exception:
            pass
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("WARNING: HF_TOKEN not found, public datasets will download without auth.")

## Config

In [ ]:
# every query and candidate is cut to MAX_LEN tokens
MAX_LEN = 512
TRAIN_BATCH, EVAL_BATCH = 8, 16
# None = use every (query, relevant candidate) pair
TRAIN_PAIR_LIMIT = None
# we retrieve the TOP_K most similar candidates per query
TOP_K = 7
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
# temperature of the contrastive loss
TEMP = 0.05
SEED = 42

RANK, ALPHA = 16, 32.0
# full = the top FULL_TOP_LAYERS blocks plus final norm train, everything else is frozen
FULL_TOP_LAYERS = 4
KD_WEIGHT = 0.5

# lr and epochs per experiment, lower epochs here if kaggle runtime becomes a problem
EXPERIMENTS = [
    {"num": 1, "name": "lora", "method": "lora", "lr": 3e-4, "epochs": 6},
    {"num": 2, "name": "lorma", "method": "lorma", "lr": 3e-4, "epochs": 6},
    # the top blocks are real weights -> lower lr
    {"num": 3, "name": "full", "method": "full", "lr": 1e-4, "epochs": 6},
    # the student has many trainable weights -> lower lr
    {"num": 4, "name": "distillation", "method": "distillation", "lr": 1e-4, "epochs": 8, "kd": True},
]

WORK_DIR = Path("/kaggle/working")
BASE_DIR = WORK_DIR / "phase7_pcr"
BASE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if USE_AMP else "")

## Phase 1 checkpoint and tokenizer

In [ ]:
# first match under /kaggle/input
def resolve_artifact(filename):
    for root, _, files in os.walk("/kaggle/input"):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Missing {filename}")


CKPT_PATH = resolve_artifact("gqa_swa_best.pt")
TOK_PATH = resolve_artifact("bpe_legal_16k_v2.pkl")
print("checkpoint:", CKPT_PATH)
print("tokenizer :", TOK_PATH)


class CustomBPETokenizer:
    def __init__(self, vocab_size=16384):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
        self.pad_token_id, self.unk_token_id, self.bos_token_id, self.eos_token_id = 0, 1, 2, 3
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\S|\s+(?!\S)|\s+""")
        self.merges, self.vocab, self.inverse_vocab, self._word_cache = {}, {}, {}, {}

    def _tokenize_word(self, word):
        word_tuple = tuple(word)
        if len(word_tuple) <= 1:
            return list(word_tuple)
        while True:
            pairs = [(word_tuple[i], word_tuple[i + 1]) for i in range(len(word_tuple) - 1)]
            valid = [p for p in pairs if p in self.merges]
            if not valid:
                break
            best = min(valid, key=lambda p: self.vocab[self.merges[p]])
            merged = self.merges[best]
            new_tuple, i = [], 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == best[0] and word_tuple[i + 1] == best[1]:
                    new_tuple.append(merged)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_tuple)
            if len(word_tuple) == 1:
                break
        return list(word_tuple)

    def encode(self, text, add_special_tokens=False):
        if not text:
            return []
        ids = []
        for word in self.pat.findall(text):
            if word not in self._word_cache:
                self._word_cache[word] = [self.vocab.get(t, self.unk_token_id) for t in self._tokenize_word(word)]
            ids.extend(self._word_cache[word])
        if add_special_tokens:
            ids = [self.bos_token_id] + ids + [self.eos_token_id]
        return ids

    @classmethod
    def load(cls, filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        tok = cls(data["target_vocab_size"])
        tok.vocab, tok.inverse_vocab, tok.merges = data["vocab"], data["inverse_vocab"], data["merges"]
        print(f"Loaded tokenizer from {filepath} (Vocab size: {len(tok.vocab)})")
        return tok


tokenizer = CustomBPETokenizer.load(TOK_PATH)

## Data

In [ ]:
REPO = "Exploration-Lab/CS-781-Capstone"
P7 = "Phase_7_PCR"


def fetch_json(rel_path):
    path = hf_hub_download(repo_id=REPO, filename=rel_path, repo_type="dataset", token=HF_TOKEN)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


print("Downloading PCR datasets...")
train_q = fetch_json(f"{P7}/Train/train.json")
train_c = fetch_json(f"{P7}/Train/train_candidates.json")
dev_q = fetch_json(f"{P7}/Dev/dev.json")
dev_c = fetch_json(f"{P7}/Dev/dev_candidates.json")
test_q = fetch_json(f"{P7}/Test/test.json")
test_c = fetch_json(f"{P7}/Test/test_candidates.json")


# some texts are lists of sentences, we want one string
def concat_text(data):
    for k, v in data.items():
        if isinstance(v["text"], list):
            v["text"] = " ".join(v["text"])
    return data


train_q, train_c = concat_text(train_q), concat_text(train_c)
dev_q, dev_c = concat_text(dev_q), concat_text(dev_c)
test_q, test_c = concat_text(test_q), concat_text(test_c)

print(f"train queries {len(train_q)} | dev queries {len(dev_q)} | test queries {len(test_q)}")

## Query and candidate pairs

In [ ]:
# one training example = a query and one of its relevant candidates
class PairDataset(Dataset):
    def __init__(self, queries, candidates, limit=None):
        self.pairs = []
        # integer ids so the loss can spot the same query or candidate twice in a batch
        q_index, c_index = {}, {}
        for q_id, q_data in queries.items():
            for c_id in q_data["relevant_candidates"]:
                if c_id in candidates:
                    qi = q_index.setdefault(q_id, len(q_index))
                    ci = c_index.setdefault(c_id, len(c_index))
                    self.pairs.append((qi, ci, q_data["text"], candidates[c_id]["text"]))
        if limit:
            self.pairs = self.pairs[:limit]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        qi, ci, q_text, c_text = self.pairs[i]
        q_ids = tokenizer.encode(q_text, add_special_tokens=True)[:MAX_LEN]
        c_ids = tokenizer.encode(c_text, add_special_tokens=True)[:MAX_LEN]
        return torch.tensor(q_ids), torch.tensor(c_ids), qi, ci


def collate_fn(batch):
    L_q = max(len(x[0]) for x in batch)
    L_c = max(len(x[1]) for x in batch)
    # pad with the pad id and mask 0
    q_ids = torch.full((len(batch), L_q), tokenizer.pad_token_id, dtype=torch.long)
    c_ids = torch.full((len(batch), L_c), tokenizer.pad_token_id, dtype=torch.long)
    q_mask = torch.zeros((len(batch), L_q), dtype=torch.long)
    c_mask = torch.zeros((len(batch), L_c), dtype=torch.long)

    for i, (q, c, _, _) in enumerate(batch):
        q_ids[i, :len(q)] = q
        q_mask[i, :len(q)] = 1
        c_ids[i, :len(c)] = c
        c_mask[i, :len(c)] = 1
    q_idx = torch.tensor([x[2] for x in batch])
    c_idx = torch.tensor([x[3] for x in batch])
    return q_ids.to(DEVICE), q_mask.to(DEVICE), c_ids.to(DEVICE), c_mask.to(DEVICE), q_idx.to(DEVICE), c_idx.to(DEVICE)


train_loader = DataLoader(PairDataset(train_q, train_c, limit=TRAIN_PAIR_LIMIT), batch_size=TRAIN_BATCH, shuffle=True, collate_fn=collate_fn, drop_last=True)
print("train pairs", len(train_loader.dataset), "| batches/epoch", len(train_loader))

## Phase 1 model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return self.weight * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6))


class SwiGLU(nn.Module):
    def __init__(self, in_f, hid_f):
        super().__init__()
        self.w1 = nn.Linear(in_f, hid_f, bias=False)
        self.w2 = nn.Linear(in_f, hid_f, bias=False)
        self.w3 = nn.Linear(hid_f, in_f, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


def apply_rope(q, k):
    T, hd = q.size(-2), q.size(-1)
    # rotary position tables
    inv_freq = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=q.device).float() / hd))
    t = torch.arange(T, device=q.device).float()
    emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
    cos, sin = emb.cos()[None, None, :], emb.sin()[None, None, :]

    def rot(x):
        return torch.cat((-x[..., hd // 2:], x[..., : hd // 2]), dim=-1)

    return q * cos + rot(q) * sin, k * cos + rot(k) * sin


# fewer k/v heads than q heads (grouped query)
class GQACausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window=None):
        super().__init__()
        self.n_heads, self.n_kv_heads, self.head_dim = n_heads, n_kv_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k)
        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, window=None):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQACausalAttention(d_model, n_heads, n_kv_heads, window)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


# phase 1 without the lm head, it returns the hidden states
class LegalEfficientTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816, window=256):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        # every 4th layer is global, the others get the window
        self.layers = nn.ModuleList([
            EfficientDecoderBlock(d_model, n_heads, n_kv_heads, d_ff, None if (i + 1) % 4 == 0 else window)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)

    def forward(self, idx):
        x = self.tok_emb(idx)
        for layer in self.layers:
            # checkpoint the blocks while training, so it fits on the gpu
            x = grad_ckpt(layer, x, use_reentrant=False) if self.training else layer(x)
        return self.final_norm(x)


ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
sd = {k.replace("_orig_mod.", "", 1): v for k, v in (ckpt.get("state_dict") or ckpt.get("model") or ckpt).items()}
cfg = ckpt.get("config", {"vocab_size": 16384, "d_model": 1024, "n_heads": 16, "n_kv_heads": 4, "n_layers": 16, "d_ff": 2816})


# a fresh copy of phase 1 every time, so experiments never share weights
# strict=False because the checkpoint also holds the lm head, which this model does not have
def load_backbone():
    m = LegalEfficientTransformer(**cfg)
    m.load_state_dict(sd, strict=False)
    return m.to(DEVICE)


log("Phase 1 model and tokenizer ready.")

## Bi-encoder

In [ ]:
# dropout is only active in training mode (so not at eval / test time)
# same idea as lora but the update acts on the output of W instead of the input
class LoRMALinear(nn.Module):
    def __init__(self, base, r=RANK, alpha=ALPHA):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        out_f = base.weight.size(0)
        self.A = nn.Parameter(torch.randn(r, out_f) * 0.02)
        # B starts at zero so at the start it behaves exactly like phase 1
        self.B = nn.Parameter(torch.zeros(out_f, r))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * (F.dropout(h, 0.05, training=self.training) @ self.A.t() @ self.B.t())


# frozen W plus a small trainable B @ A
class LoRALinear(nn.Module):
    def __init__(self, base, r=RANK, alpha=ALPHA):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base.weight.size(1)) * 0.02)
        self.B = nn.Parameter(torch.zeros(base.weight.size(0), r))
        self.scale = alpha / r

    def forward(self, x):
        # W x + scale * B A x
        return self.base(x) + self.scale * (F.dropout(x, 0.05, training=self.training) @ self.A.t() @ self.B.t())


# mean pooling over the real tokens, then l2 norm, so a dot product of two embeddings is a cosine similarity
class BiEncoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, ids, attention_mask):
        hs = self.backbone(ids).float()
        mask = attention_mask.unsqueeze(-1).to(hs.dtype)
        emb = (hs * mask).sum(1) / mask.sum(1).clamp(min=1e-8)
        return F.normalize(emb, p=2, dim=1)


# wrap the target linears for the method
def make_model(method):
    m = load_backbone()

    # for lora and lorma the backbone is frozen first, only the adapters and the norms train
    if method in ("lora", "lorma"):
        for p in m.parameters():
            p.requires_grad_(False)

    if method == "lora":
        # attention projections and the swiglu mlp
        for b in m.layers:
            for n in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(b.attn, n, LoRALinear(getattr(b.attn, n)))
            for n in ("w1", "w2", "w3"):
                setattr(b.ffn, n, LoRALinear(getattr(b.ffn, n)))
            b.attn_norm.requires_grad_(True)
            b.ffn_norm.requires_grad_(True)
            m.final_norm.requires_grad_(True)

    elif method == "lorma":
        # attention projections only
        for b in m.layers:
            for n in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(b.attn, n, LoRMALinear(getattr(b.attn, n)))
            b.attn_norm.requires_grad_(True)
            m.final_norm.requires_grad_(True)

    elif method == "full":
        # freeze everything, then open the top blocks and the final norm
        for p in m.parameters():
            p.requires_grad_(False)
        for b in m.layers[-FULL_TOP_LAYERS:]:
            b.requires_grad_(True)
        m.final_norm.requires_grad_(True)

    elif method == "distillation":
        # small student: every second phase 1 layer (8 of 16), lora on the attention projections
        m.layers = nn.ModuleList([m.layers[i] for i in range(0, len(m.layers), 2)])
        for b in m.layers:
            for n in ("q_proj", "k_proj", "v_proj", "o_proj"):
                setattr(b.attn, n, LoRALinear(getattr(b.attn, n)))

    else:
        raise ValueError(f"unknown method: {method}")

    model = BiEncoder(m).to(DEVICE)
    tr_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot_params = sum(p.numel() for p in model.parameters())
    print(f"[{method.upper()}] trainable {tr_params / 1e6:.2f}M / total {tot_params / 1e6:.2f}M ({tr_params / tot_params:.1%})")
    return model

## Evaluation and submission

In [ ]:
# one normalised embedding for every text in the dict (queries or candidates)
@torch.no_grad()
def get_embeddings(model, data_dict):
    model.eval()
    keys = list(data_dict.keys())
    embs = []
    for i in tqdm(range(0, len(keys), EVAL_BATCH), desc="Encoding", leave=False):
        batch_keys = keys[i:i + EVAL_BATCH]
        ids_batch = []
        for k in batch_keys:
            ids_batch.append(torch.tensor(tokenizer.encode(data_dict[k]["text"], add_special_tokens=True)[:MAX_LEN]))
        L = max(len(x) for x in ids_batch)
        ids = torch.full((len(ids_batch), L), tokenizer.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(ids_batch), L), dtype=torch.long)
        for j, seq in enumerate(ids_batch):
            ids[j, :len(seq)] = seq
            mask[j, :len(seq)] = 1

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            emb = model(ids.to(DEVICE), mask.to(DEVICE))
        embs.append(emb.float().cpu())
    return keys, torch.cat(embs, dim=0)


# micro f1 / precision / recall of the TOP_K candidates per query on the dev set
def evaluate_dev(model):
    c_keys, c_embs = get_embeddings(model, dev_c)
    q_keys, q_embs = get_embeddings(model, dev_q)

    # cosine similarity matrix
    sim = torch.matmul(q_embs, c_embs.t())
    _, top_indices = torch.topk(sim, k=TOP_K, dim=-1)

    true_positives = 0
    total_relevant = 0
    total_retrieved = 0

    for i, q_id in enumerate(q_keys):
        retrieved = [c_keys[idx] for idx in top_indices[i].tolist()]
        relevant = dev_q[q_id]["relevant_candidates"]
        true_positives += len(set(retrieved) & set(relevant))
        total_relevant += len(relevant)
        total_retrieved += TOP_K

    p = true_positives / total_retrieved if total_retrieved > 0 else 0
    r = true_positives / total_relevant if total_relevant > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return {"Micro-F1@7": f1, "Precision@7": p, "Recall@7": r}


# the TOP_K most similar test candidates for every test query, zipped as prediction.json
def generate_submission(model, exp_num, method_name):
    log(f"generating test predictions for exp {exp_num}...")
    c_keys, c_embs = get_embeddings(model, test_c)
    q_keys, q_embs = get_embeddings(model, test_q)

    sim = torch.matmul(q_embs, c_embs.t())
    _, top_indices = torch.topk(sim, k=TOP_K, dim=-1)

    preds = {}
    for i, q_id in enumerate(q_keys):
        preds[q_id] = [c_keys[idx] for idx in top_indices[i].tolist()]

    pred_path = BASE_DIR / f"prediction_{method_name}.json"
    zip_name = WORK_DIR / f"submission_exp{exp_num}_{method_name}.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(preds, f, indent=2)

    # prediction.json has to be at the root of the zip
    with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")

    print(f"wrote {zip_name} | {len(preds)} queries")
    return zip_name

## Training loop

In [ ]:
# in-batch similarity logits, the same query or the same candidate twice in a batch (off the diagonal) is a false negative, so it is masked out
def masked_logits(q_emb, c_emb, q_idx, c_idx, temp=TEMP):
    sim = torch.matmul(q_emb.float(), c_emb.float().t()) / temp
    eye = torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
    false_neg = ((q_idx[:, None] == q_idx[None, :]) | (c_idx[:, None] == c_idx[None, :])) & ~eye
    return sim.masked_fill(false_neg, -1e9)


# symmetric infonce, query -> candidate and candidate -> query
def infonce_loss(q_emb, c_emb, q_idx, c_idx, temp=TEMP):
    sim = masked_logits(q_emb, c_emb, q_idx, c_idx, temp)
    labels = torch.arange(sim.size(0), device=sim.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))


# only the weights that train are kept for the best epoch
def snapshot(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in state:
                p.copy_(state[n].to(p.device))


# trains for the epochs of the experiment, prints dev scores every epoch and puts the best dev epoch back at the end
# teacher given = distillation (infonce plus a kl on the similarity logits of the teacher)
def run_training(model, exp, teacher=None):
    tag, epochs = exp["name"], exp["epochs"]
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=exp["lr"], weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    # warmup, then cosine down to 1% of the lr
    total = epochs * len(train_loader)
    warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (s + 1) / warm if s < warm
        else max(0.01, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    )

    best_f1, best_state, best_ep = -1.0, None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        bar = tqdm(train_loader, desc=f"{tag} ep{ep}/{epochs}", leave=False)

        for q_ids, q_mask, c_ids, c_mask, q_idx, c_idx in bar:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                q_emb = model(q_ids, q_mask)
                c_emb = model(c_ids, c_mask)
                loss = infonce_loss(q_emb, c_emb, q_idx, c_idx)

                if teacher is not None:
                    with torch.no_grad():
                        t_q_emb = teacher(q_ids, q_mask)
                        t_c_emb = teacher(c_ids, c_mask)
                    # kl on the similarity logits
                    s_sim = masked_logits(q_emb, c_emb, q_idx, c_idx)
                    t_sim = masked_logits(t_q_emb, t_c_emb, q_idx, c_idx)
                    kd_loss = F.kl_div(F.log_softmax(s_sim, dim=-1), F.softmax(t_sim, dim=-1), reduction="batchmean")
                    loss = KD_WEIGHT * loss + (1 - KD_WEIGHT) * kd_loss

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            sched.step()
            train_loss += loss.item()
            bar.set_postfix(loss=f"{train_loss / (bar.n + 1):.4f}")

        metrics = evaluate_dev(model)
        print(f"  [{tag}] ep {ep}: loss {train_loss / len(train_loader):.4f} | dev Micro-F1@7 {metrics['Micro-F1@7']:.4f} "
              f"P@7 {metrics['Precision@7']:.4f} R@7 {metrics['Recall@7']:.4f} | {(time.time() - t0) / 60:.1f} min")

        if metrics["Micro-F1@7"] > best_f1:
            best_f1, best_state, best_ep = metrics["Micro-F1@7"], snapshot(model), ep

    print(f"  [{tag}] restoring best epoch {best_ep} (dev Micro-F1@7 {best_f1:.4f})")
    restore(model, best_state)
    return best_f1

## Train

In [ ]:
teacher = None

for exp in EXPERIMENTS:
    print(f"\n===== exp {exp['num']}: {exp['name']} =====")

    model = make_model(exp["method"])
    # distillation uses the full run (exp 3) as the teacher
    run_training(model, exp, teacher=teacher if exp.get("kd") else None)
    generate_submission(model, exp["num"], exp["name"])

    if exp["method"] == "full":
        # keep it in memory as the teacher for exp 4
        teacher = model
        teacher.eval()
    else:
        del model
    gc.collect(); torch.cuda.empty_cache()

log("all experiments done.")